# E-Commerce Sales & Profit Analysis
## Data Import to MSSQL

This notebook imports the cleaned CSV data into Microsoft SQL Server.

### 1. Import Required Libraries

In [1]:
import os
import pandas as pd

from sqlalchemy import create_engine, text

from urllib.parse import quote_plus

### 2. Configuration

In [2]:
# SQL Server Configuration

SERVER = "localhost"
DATABASE = "ecommerce_analytics"
DRIVER = "ODBC Driver 17 for SQL Server"

CONNECTION_STRING = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Configuration loaded successfully!")
print(f"Server   : {SERVER}")
print(f"Database : {DATABASE}")
print(f"Driver   : {DRIVER}")
print("Auth     : Windows Authentication")

Configuration loaded successfully!
Server   : localhost
Database : ecommerce_analytics
Driver   : ODBC Driver 17 for SQL Server
Auth     : Windows Authentication


### 3. Create SQLAlchemy Engine

In [3]:
# Create SQLAlchemy Engine

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(CONNECTION_STRING)
)

engine = create_engine(
    connection_url,
    fast_executemany=True
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


### 4. Test SQL Server Connection

In [4]:
# Test Connection

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name")
        )

        row = result.fetchone()

        print("✅ SQL Server connection successful!")
        print(f"Server   : {row.server_name}")
        print(f"Database : {row.database_name}")

except Exception as e:
    print("❌ Connection failed!")
    print(f"Error: {e}")

C:\Users\dheer\AppData\Local\Temp\ipykernel_3724\128254089.py:4: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as connection:


✅ SQL Server connection successful!
Server   : Mahour-Ji
Database : ecommerce_analytics


### 5. Check CSV File

In [5]:
# CSV Configuration

CSV_PATH = "../data/processed/ecommerce_clean.csv"

if os.path.exists(CSV_PATH):
    file_size = os.path.getsize(CSV_PATH) / (1024 * 1024)

    print("✅ CSV file found!")
    print(f"Path : {CSV_PATH}")
    print(f"Size : {file_size:.2f} MB")

else:
    raise FileNotFoundError(
        f"❌ CSV file not found: {CSV_PATH}"
    )

✅ CSV file found!
Path : ../data/processed/ecommerce_clean.csv
Size : 2.78 MB


### 6. Read CSV File

In [6]:
# Read Cleaned CSV

CSV_PATH = '../data/processed/ecommerce_clean.csv'

print("Reading CSV file...")

df = pd.read_csv(CSV_PATH)

print("✅ CSV loaded successfully!")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

Reading CSV file...
✅ CSV loaded successfully!
Rows    : 9,994
Columns : 32


In [7]:
# check head

df.head()

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Quarter,Month,Month_Name,Year_Month,Week,Day_of_Week,Order_to_Ship_Days,Profit_Margin,Sales_per_Quantity,Discount_Band
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,16.00,130.9800,0%
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,30.00,243.9800,0%
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,2,6,June,2016-06,23,Sunday,4,47.00,7.3100,0%
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,-40.00,191.5155,30%+
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,11.25,11.1840,11-20%


### 7. Inspect Columns

In [8]:
# Original Columns

print("Original columns:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

Original columns:
 1. Row_ID
 2. Order_ID
 3. Order_Date
 4. Ship_Date
 5. Ship_Mode
 6. Customer_ID
 7. Customer_Name
 8. Segment
 9. Country
10. City
11. State
12. Postal_Code
13. Region
14. Product_ID
15. Category
16. Sub-Category
17. Product_Name
18. Sales
19. Quantity
20. Discount
21. Profit
22. Year
23. Quarter
24. Month
25. Month_Name
26. Year_Month
27. Week
28. Day_of_Week
29. Order_to_Ship_Days
30. Profit_Margin
31. Sales_per_Quantity
32. Discount_Band


### 8. Rename Columns to Match SQL Table

In [9]:
# Rename Columns to Match SQL Server Table

column_mapping = {
    "Row_ID": "row_id",
    "Order_ID": "order_id",
    "Order_Date": "order_date",
    "Ship_Date": "ship_date",
    "Ship_Mode": "ship_mode",
    "Customer_ID": "customer_id",
    "Customer_Name": "customer_name",
    "Segment": "segment",
    "Country": "country",
    "City": "city",
    "State": "state",
    "Postal_Code": "postal_code",
    "Region": "region",
    "Product_ID": "product_id",
    "Category": "category",
    "Sub-Category": "sub_category",
    "Product_Name": "product_name",
    "Sales": "sales",
    "Quantity": "quantity",
    "Discount": "discount",
    "Profit": "profit",
    "Year": "year",
    "Quarter": "quarter",
    "Month": "month",
    "Month_Name": "month_name",
    "Year_Month": "year_month",
    "Week": "week",
    "Day_of_Week": "day_of_week",
    "Order_to_Ship_Days": "order_to_ship_days",
    "Profit_Margin": "profit_margin",
    "Sales_per_Quantity": "sales_per_quantity",
    "Discount_Band": "discount_band"
}

df.rename(columns=column_mapping, inplace=True)

print("✅ Column mapping completed!")

✅ Column mapping completed!


In [10]:
# Check renamed columns

print("Renamed columns:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:2}. {column}")

Renamed columns:
 1. row_id
 2. order_id
 3. order_date
 4. ship_date
 5. ship_mode
 6. customer_id
 7. customer_name
 8. segment
 9. country
10. city
11. state
12. postal_code
13. region
14. product_id
15. category
16. sub_category
17. product_name
18. sales
19. quantity
20. discount
21. profit
22. year
23. quarter
24. month
25. month_name
26. year_month
27. week
28. day_of_week
29. order_to_ship_days
30. profit_margin
31. sales_per_quantity
32. discount_band


### 9. Validate Required Columns

In [11]:
# Validate Required Columns

required_columns = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit",
    "year",
    "quarter",
    "month",
    "month_name",
    "year_month",
    "week",
    "day_of_week",
    "order_to_ship_days",
    "profit_margin",
    "sales_per_quantity",
    "discount_band"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Missing columns: {missing_columns}"
    )

print("✅ All required columns are present!")

✅ All required columns are present!


### 10. Convert Data Types

In [12]:
# Data Type Conversion

# Date columns
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    errors="coerce"
)

# Integer columns
integer_columns = [
    "row_id",
    "quantity",
    "year",
    "quarter",
    "month",
    "week",
    "order_to_ship_days"
]

for column in integer_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    ).astype("Int64")

# Decimal / numeric columns
numeric_columns = [
    "sales",
    "discount",
    "profit",
    "profit_margin",
    "sales_per_quantity"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("✅ Data types converted!")

✅ Data types converted!


In [13]:
# check data types
df.dtypes

row_id                         Int64
order_id                      object
order_date            datetime64[ns]
ship_date             datetime64[ns]
ship_mode                     object
customer_id                   object
customer_name                 object
segment                       object
country                       object
city                          object
state                         object
postal_code                    int64
region                        object
product_id                    object
category                      object
sub_category                  object
product_name                  object
sales                        float64
quantity                       Int64
discount                     float64
profit                       float64
year                           Int64
quarter                        Int64
month                          Int64
month_name                    object
year_month                    object
week                           Int64
d

### 11. Handle Missing Values

In [14]:
# Convert Pandas Missing Values to SQL NULL

df = df.astype(object).where(
    pd.notna(df),
    None
)

print("✅ Missing values prepared for SQL Server!")

✅ Missing values prepared for SQL Server!


### 12. Data Quality Check

In [15]:
# Data Quality Check

print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

print(f"Total rows       : {len(df):,}")
print(f"Total columns    : {len(df.columns)}")
print(f"Duplicate rows   : {df.duplicated().sum():,}")

print("\nMissing values:")
print(
    df.isnull()
        .sum()
        .loc[lambda x: x > 0]
)

DATA QUALITY CHECK
Total rows       : 9,994
Total columns    : 32
Duplicate rows   : 0

Missing values:
Series([], dtype: int64)


### 13. Primary Key Validation

In [16]:
# Validate row_id

null_row_ids = df["row_id"].isnull().sum()
duplicate_row_ids = df["row_id"].duplicated().sum()

print(f"NULL row_id values       : {null_row_ids}")
print(f"Duplicate row_id values  : {duplicate_row_ids}")

if null_row_ids > 0:
    raise ValueError("❌ row_id contains NULL values!")

if duplicate_row_ids > 0:
    raise ValueError("❌ row_id contains duplicate values!")

print("✅ row_id validation passed!")

NULL row_id values       : 0
Duplicate row_id values  : 0
✅ row_id validation passed!


### 14. Display Data Before Import

In [17]:
# Preview Data

display(df.head(10))

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,quarter,month,month_name,year_month,week,day_of_week,order_to_ship_days,profit_margin,sales_per_quantity,discount_band
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,16.0,130.98,0%
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,30.0,243.98,0%
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,2,6,June,2016-06,23,Sunday,4,47.0,7.31,0%
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,-40.0,191.5155,30%+
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,11.25,11.184,11-20%
5,6,CA-2014-115812,2014-06-09 00:00:00,2014-06-14 00:00:00,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,29.0,6.98,0%
6,7,CA-2014-115812,2014-06-09 00:00:00,2014-06-14 00:00:00,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,27.0,1.82,0%
7,8,CA-2014-115812,2014-06-09 00:00:00,2014-06-14 00:00:00,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,10.0,151.192,11-20%
8,9,CA-2014-115812,2014-06-09 00:00:00,2014-06-14 00:00:00,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,31.25,6.168,11-20%
9,10,CA-2014-115812,2014-06-09 00:00:00,2014-06-14 00:00:00,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,30.0,22.98,0%


### 15. Check Existing SQL Server Data

In [18]:
# Check Existing Data

with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*) AS total_rows
            FROM dbo.ecommerce_sales
        """)
    )

    existing_rows = result.scalar()

print(f"Existing rows in SQL Server: {existing_rows:,}")

Existing rows in SQL Server: 0


### 16. Choose Import Mode

In [19]:
# Import Configuration

TABLE_NAME = "ecommerce_sales"
SCHEMA_NAME = "dbo"

if existing_rows > 0:
    print(
        "⚠️ Table already contains data."
    )
    print(
        "Skipping automatic import to prevent duplicate rows."
    )
else:
    print("Table is empty. Ready for import.")

Table is empty. Ready for import.


### 17. Import Data into SQL Server

In [20]:
# Import Data

if existing_rows == 0:

    print("Starting data import...")
    print(f"Rows to import: {len(df):,}")

    try:

        df.to_sql(
            name=TABLE_NAME,
            con=engine,
            schema=SCHEMA_NAME,
            if_exists="append",
            index=False,
            chunksize=1000,
            method=None
        )

        print("✅ Data import completed successfully!")

    except Exception as e:

        print("❌ Data import failed!")
        print(f"Error: {e}")

else:

    print("⏭️ Import skipped because data already exists.")

Starting data import...
Rows to import: 9,994
✅ Data import completed successfully!


### 18. Verify Imported Row Count

In [21]:
# Verify Row Count

with engine.connect() as connection:

    result = connection.execute(
        text("""
            SELECT COUNT(*) AS total_rows
            FROM dbo.ecommerce_sales
        """)
    )

    sql_row_count = result.scalar()

print(f"CSV rows        : {len(df):,}")
print(f"SQL Server rows : {sql_row_count:,}")

if sql_row_count == len(df):
    print("✅ Row count verification passed!")

else:
    print("⚠️ Row count does not match!")

CSV rows        : 9,994
SQL Server rows : 9,994
✅ Row count verification passed!


### 19. Verify Business Metrics

In [22]:
# Verify Business Metrics

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers,
    COUNT(DISTINCT product_id) AS total_products,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    SUM(quantity) AS total_quantity
FROM dbo.ecommerce_sales;
"""

verification_df = pd.read_sql(
    query,
    engine
)

display(verification_df)

,total_rows,total_orders,total_customers,total_products,total_sales,total_profit,total_quantity
0,9994,5009,793,1862,2297201.05,286397.78,37873


### 20. Check Date Range

In [23]:
# Date Range Verification

query = """
SELECT
    MIN(order_date) AS first_order_date,
    MAX(order_date) AS last_order_date
FROM dbo.ecommerce_sales;
"""

date_df = pd.read_sql(
    query,
    engine
)

display(date_df)

,first_order_date,last_order_date
0,2014-01-03,2017-12-30


### 21. Retrieve Sample Data

In [24]:
# Sample Data from SQL Server

query = """
SELECT TOP 10 *
FROM dbo.ecommerce_sales
ORDER BY row_id;
"""

df_sample = pd.read_sql(
    query,
    engine
)

display(df_sample)

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,quarter,month,month_name,year_month,week,day_of_week,order_to_ship_days,profit_margin,sales_per_quantity,discount_band
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,16.00,130.98,0%
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,4,11,November,2016-11,45,Tuesday,3,30.00,243.98,0%
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,2,6,June,2016-06,23,Sunday,4,47.00,7.31,0%
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,-40.00,191.52,30%+
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,4,10,October,2015-10,41,Sunday,7,11.25,11.18,11-20%
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,29.00,6.98,0%
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,27.00,1.82,0%
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,10.00,151.19,11-20%
8,9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,31.25,6.17,11-20%
9,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,2,6,June,2014-06,24,Monday,5,30.00,22.98,0%


### 22. Run First Analytical Query

In [25]:
# Category Performance

query = """
SELECT
    category,
    SUM(sales) AS total_sales,
    SUM(profit) AS total_profit,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(quantity) AS total_quantity,
    CAST(
        SUM(profit) * 100.0 /
        NULLIF(SUM(sales), 0)
        AS DECIMAL(10,2)
    ) AS profit_margin_pct
FROM dbo.ecommerce_sales
GROUP BY category
ORDER BY total_sales DESC;
"""

category_df = pd.read_sql(
    query,
    engine
)

display(category_df)

,category,total_sales,total_profit,total_orders,total_quantity,profit_margin_pct
0,Technology,836154.10,145455.66,1544,6939,17.40
1,Furniture,741999.97,18451.25,1764,8028,2.49
2,Office Supplies,719046.98,122490.87,3742,22906,17.04


### 23. Final Summary

In [26]:
# Import Summary

print("=" * 70)
print("E-COMMERCE DATA IMPORT SUMMARY")
print("=" * 70)

print(f"CSV File          : {CSV_PATH}")
print(f"Database          : {DATABASE}")
print(f"Server            : {SERVER}")
print(f"Table             : {SCHEMA_NAME}.{TABLE_NAME}")
print(f"CSV Rows          : {len(df):,}")
print(f"SQL Server Rows   : {sql_row_count:,}")

if sql_row_count == len(df):
    print("Status            : ✅ SUCCESS")
else:
    print("Status            : ⚠️ CHECK DATA")

print("=" * 70)
print("Data is ready for SQL analysis and Power BI!")
print("=" * 70)

E-COMMERCE DATA IMPORT SUMMARY
CSV File          : ../data/processed/ecommerce_clean.csv
Database          : ecommerce_analytics
Server            : localhost
Table             : dbo.ecommerce_sales
CSV Rows          : 9,994
SQL Server Rows   : 9,994
Status            : ✅ SUCCESS
Data is ready for SQL analysis and Power BI!
